In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# holt winters using 3 components (Level, Trend, Seasonal) to forecast sales
# - Level: current average sales baseline
# - Trend: growth/decline pattern over time  
# - Seasonal: weekly patterns (seasonal_periods=7 for day-of-week effects)

# Using additive mode since we apply log transformation to stabilize variance

def forecast_holt_winters(group_df):
    x = group_df.copy()
    if x['sales'].notna().sum() > 30:
        try:
            sales_log = np.log1p(x['sales'].dropna())
            
            model = ExponentialSmoothing(
                sales_log,
                trend='add',
                seasonal='add',
                seasonal_periods=7
            ).fit(optimized=True)
            
            forecast_len = x['sales'].isna().sum()
            
            if forecast_len > 0:
                forecast_log = model.forecast(forecast_len)
                x.loc[x['sales'].isna(), 'sales'] = np.expm1(forecast_log).clip(lower=0).values
            
            return x
        except:
            x['sales'] = x['sales'].bfill().ffill()
            return x
    else:
        x['sales'] = x['sales'].bfill().ffill()
        return x

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

df_train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
df_test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')
df_sample_subm = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv')

# -------------------------------
# 3️⃣ Préparer les données
# -------------------------------
df_train = df_train[['id','date', 'store_nbr', 'family', 'sales']]
df_train['source'] = 'train'

df_test = df_test[['id','date', 'store_nbr', 'family']]
df_test['source'] = 'test'

df_concat = pd.concat([df_train, df_test], ignore_index=True)
df_concat['date'] = pd.to_datetime(df_concat['date'])
df_concat = df_concat.sort_values(['store_nbr', 'family', 'date'])


In [ ]:
def cap_outliers(x):
    low = 0
    high = x.quantile(0.99)
    return x.clip(lower=low, upper=high)

df_outliers_cap = df_concat.copy()
df_outliers_cap['sales'] = df_concat.groupby(['store_nbr', 'family'])['sales'].transform(cap_outliers)


In [ ]:
df_outliers_cap = (
    df_outliers_cap
    .groupby(['store_nbr', 'family'], group_keys=False)
    .apply(forecast_holt_winters)
)

df_test_final = df_outliers_cap[df_outliers_cap['source'] == 'test']
df_test_final = df_test_final[['id', 'sales']]
df_test_final = df_test_final.sort_values(['id'])

assert (df_test_final['id'].values == df_sample_subm['id'].values).all(), "IDs do not match or are in different order"


df_test_final.to_csv("submission.csv", index=False)
print(df_test_final) 
print("Submission file created: submission.csv")